In [66]:
from IPython.core.interactiveshell import InteractiveShell
InteractiveShell.ast_node_interactivity = "all"
import pandas as pd
import numpy as np
df = pd.read_csv("../data/social_posts.csv")

In [67]:
df.groupby("brand")["likes"].mean()

brand
FitFuel        1179.357143
Glowly         1639.629032
HomeNest        459.227273
SnackBox       2688.783333
UrbanThread     841.910714
Name: likes, dtype: float64

df.groupby( GROUP_COLUMN )[ VALUE_COLUMN ].FUNCTION()
Always pick the value column. If you leave it out, pandas tries to average text columns like post_id, and you get an error.

In [68]:
avg = df.groupby("brand")["likes"].mean().round(1)
avg.sort_values(ascending=False)   # rank brands
avg["Glowly"]                      # one value by label → 1639.6
avg.idxmax()                       # label of the biggest → 'SnackBox'

brand
SnackBox       2688.8
Glowly         1639.6
FitFuel        1179.4
UrbanThread     841.9
HomeNest        459.2
Name: likes, dtype: float64

np.float64(1639.6)

'SnackBox'

In [69]:
df.groupby("brand").size()                 # rows per brand (NaN or not)
df.groupby("brand")["comments"].count()    # NON-missing comments per brand
# The difference is the missing comments. size counts rows, and count counts non-missing values.

brand
FitFuel        56
Glowly         62
HomeNest       66
SnackBox       60
UrbanThread    56
dtype: int64

brand
FitFuel        53
Glowly         61
HomeNest       64
SnackBox       60
UrbanThread    52
Name: comments, dtype: int64

In [70]:
df.groupby("platform")["likes"].agg(["mean", "median", "max"])
# You pass function names as strings in a list. The result is a DataFrame with one column per stat.

,mean,median,max
platform,,,
Facebook,526.718750,399.5,2304
Instagram,1151.871212,853.0,6133
TikTok,2122.375000,1449.5,13577


In [71]:
df.groupby("brand").agg(
    n_posts=("post_id", "count"),
    avg_likes=("likes", "mean"),
    total_shares=("shares", "sum"),
)
# new_name = (column, function)

,n_posts,avg_likes,total_shares
brand,,,
FitFuel,56,1179.357143,1230.0
Glowly,62,1639.629032,1773.0
HomeNest,66,459.227273,547.0
SnackBox,60,2688.783333,2949.0
UrbanThread,56,841.910714,842.0


In [72]:
df.groupby(["platform", "post_type"])["likes"].mean()

platform   post_type
Facebook   carousel      484.428571
           image         474.962963
           video         613.217391
Instagram  carousel      922.166667
           image         851.160000
           video        1573.538462
TikTok     carousel     2445.925926
           image        1294.046512
           video        2913.029412
Name: likes, dtype: float64

In [73]:
df.groupby(["platform", "post_type"])["likes"].mean().reset_index()
# The index now has two levels (platform, then post_type). This is called a MultiIndex. To make it a normal flat table, add .reset_index():

,platform,post_type,likes
0,Facebook,carousel,484.428571
1,Facebook,image,474.962963
2,Facebook,video,613.217391
3,Instagram,carousel,922.166667
4,Instagram,image,851.160000
5,Instagram,video,1573.538462
6,TikTok,carousel,2445.925926
7,TikTok,image,1294.046512
8,TikTok,video,2913.029412


In [74]:
# transform: put the group answer back on every row
# .mean() shrinks 300 rows down to 5 (one per brand). Sometimes you want the brand average next to each post so you can compare them. That’s what transform does:

df["brand_avg"] = df.groupby("brand")["likes"].transform("mean")
# The result has the same length as df (300), so it can be a new column. Now you can ask “how did this post do compared to its own brand?”:
df["vs_brand"] = (df["likes"] / df["brand_avg"]).round(2)   # 2.68 = 2.68× its brand's usual

In [75]:
# .mean() / .agg()	one row per group	summary tables
# .transform("mean")	one row per original row	new columns, comparing a row to its group

## 2.6 Cheat sheet: groupby

| I want… | Write |
|---|---|
| one stat per group | `df.groupby("g")["v"].mean()` |
| rank groups | `...mean().sort_values(ascending=False)` |
| label of the top group | `...mean().idxmax()` |
| rows per group | `df.groupby("g").size()` |
| non-missing values per group | `df.groupby("g")["v"].count()` |
| proportion True per group | `df.groupby("g")["flag"].mean()` |
| several stats, one column | `df.groupby("g")["v"].agg(["mean","max"])` |
| named stats, any columns | `df.groupby("g").agg(new=("col","func"), ...)` |
| two grouping columns | `df.groupby(["g1","g2"])["v"].mean()` |
| flatten to normal columns | `.reset_index()` |
| group value on every row | `df.groupby("g")["v"].transform("mean")` |

**Rules**
- Always pick the value column before the function
- size = rows, count = non-missing
- agg shrinks to one row per group; transform keeps every row

In [76]:
toy = pd.DataFrame({
    "brand":     ["A", "B", "A", "B", "A"],
    "likes":     [10, 40, 30, 20, 20],
    "sponsored": [True, False, False, False, True],
})

In [77]:
toy.groupby("brand")["likes"].mean() #A: 20, B: 30
toy.groupby("brand")["likes"].sum() #A: 60, B: 60
toy.groupby("brand").size() #A: 3, B: 2
toy.groupby("brand")["likes"].max() #A: 30, B: 40
toy.groupby("brand")["sponsored"].mean() #A: 0.667, B: 0.000 (round in your head to 3 decimals)
toy.groupby("brand")["likes"].mean()["B"] # 30.0
toy.groupby("brand")["likes"].transform("mean") #give all 5 values in order
# 20, 30, 20, 30, 20
toy.groupby("brand")["likes"].agg(["min", "max"]).shape #(2,2)
toy.groupby(["brand", "sponsored"]).size() #3 a--t,f, b--f  how many rows does the result have?
toy.groupby("brand")["likes"].sum().idxmax() #a==b ? it will pick first in list? careful, check your answer to Q2.

brand
A    20.0
B    30.0
Name: likes, dtype: float64

brand
A    60
B    60
Name: likes, dtype: int64

brand
A    3
B    2
dtype: int64

brand
A    30
B    40
Name: likes, dtype: int64

brand
A    0.666667
B    0.000000
Name: sponsored, dtype: float64

np.float64(30.0)

0    20.0
1    30.0
2    20.0
3    30.0
4    20.0
Name: likes, dtype: float64

(2, 2)

brand  sponsored
A      False        1
       True         2
B      False        2
dtype: int64

'A'

In [83]:
df = pd.read_csv("../data/social_posts.csv")

# B1. Likes per brand

# (a) Mean likes per brand.
df.groupby("brand")['likes'].mean().sort_values(ascending=False).round(1)
# (b) Round to 1.
# (c) Sort highest first.
# (d) Write the top brand’s name as a comment.
    # SnackBox

# B2. Counting

# (a) Number of posts per platform using .size().
df.groupby('platform').size()
# (b) Number of non-missing shares per platform using .count().
df.groupby('platform')['shares'].count()
# (c) One comment: why are (a) and (b) different?
#not getting different answers? means no missing rows?

# B3. Sponsored share per brand

# (a) Proportion of sponsored posts per brand.
df.groupby('brand')['sponsored'].mean().round(3)
df.groupby('brand')['sponsored'].mean().round(3).idxmax()
# (b) Round to 3.
# (c) Use idxmax() to get the brand with the highest proportion.

# B4. Several stats

# (a) For each platform, get the mean, median and max of likes with .agg([...]).
df.groupby('platform')['likes'].agg(['mean', 'median', 'max']).round(1)
# (b) Round to 1.
# (c) One comment: for every platform the mean is bigger than the median. What does that tell you? (2.3 recap.)
#outlying likes in each brand, not indicative of general post performance

# B5. Named aggregation: a brand report

# (a) One table per brand with these columns:

brand_report=df.groupby('brand').agg(n_posts= ('post_id', 'count'), avg_likes= ('likes', 'mean'), total_shares= ('shares', 'sum'), avg_followers= ('followers', 'mean') ).sort_values('avg_likes', ascending=False).round(1)
brand_report
# n_posts = count of post_id
# avg_likes = mean of likes
# total_shares = sum of shares
# avg_followers = mean of followers
# (b) Round to 1.
# (c) Sort by avg_likes, highest first.
# (d) Store it as brand_report.

# B6. Two keys

# (a) Mean likes per platform and post_type.
df.groupby(['platform', 'post_type'])['likes'].mean().round(1).reset_index().sort_values('likes', ascending=False).head(3)
# (b) Round to 1.
# (c) Use .reset_index() to flatten it.
# (d) Sort by likes, highest first.
# (e) Show only the top 3 rows.
# (f) Comment: which platform + post_type combination wins?
    # TikTok, video

# B7. Engagement rate per brand

# (a) Make eng_rate = (likes + comments + shares) / followers × 100 (don’t round yet).
df['eng_rate']= (df['likes']+df['comments']+df['shares'])/df['followers']*100
# (b) Mean eng_rate per brand.
df.groupby('brand')['eng_rate'].mean().sort_values(ascending=False).round(2)
# (c) Round to 2.
# (d) Sort highest first.
# (e) Comment: compare this ranking to B1. Is the top brand by likes also the top by eng_rate? Why might they differ? (Hint: look at avg_followers in B5.)
# Snackbox in every case--highest avg followers, mean likes and eng_rate
# The top brand is the same ✓, but the question asked why rankings might differ, and they do. UrbanThread jumps from 4th (likes) to 2nd (eng_rate). eng_rate divides by followers, and UrbanThread has only ~42k followers, so its likes are a bigger share of its audience. Raw likes reward big accounts, and eng_rate shows who engages their audience best.

# B8. transform: compare each post to its brand

# (a) Make brand_avg = each post’s brand mean likes, using transform.
df['brand_avg']=df.groupby('brand')['likes'].transform('mean')
# (b) Make vs_brand = likes / brand_avg.
df['vs_brand']= (df['likes']/ df['brand_avg']).round(2)
# (c) Round vs_brand to 2.
# (d) Show post_id, brand, likes and vs_brand for the top 3 posts by vs_brand.
df.sort_values(by='vs_brand', ascending=False).head(3)[['post_id', 'brand', 'likes', 'vs_brand']]
# (e) Comment: one of the top 3 is not from the biggest brand. Which one, and why does vs_brand find it when plain likes wouldn’t?
	# P0180 by	HomeNest-- it outperformed because it may have been a viral post compared to its own brand average. total likes just sort by likes count but do not show relative performance of each post


# B9. Sponsored vs not, within each brand

# (a) Mean likes grouped by brand and sponsored.
df.groupby(['brand', 'sponsored'])['likes'].mean().round(1)
# (b) Round to 1.
# (c) Look at SnackBox: which is higher, sponsored or not?
#not sponsored
# (d) Look at Glowly: which is higher?
#sponsored
# (e) Comment: in 2.4 (B7) sponsored posts had fewer likes overall. Is that true for every brand?
#not true for fitfuel and glowly-- their sponsored posts perform better

brand
SnackBox       2688.8
Glowly         1639.6
FitFuel        1179.4
UrbanThread     841.9
HomeNest        459.2
Name: likes, dtype: float64

platform
Facebook      64
Instagram    132
TikTok       104
dtype: int64

platform
Facebook      62
Instagram    125
TikTok       103
Name: shares, dtype: int64

brand
FitFuel        0.286
Glowly         0.177
HomeNest       0.288
SnackBox       0.217
UrbanThread    0.250
Name: sponsored, dtype: float64

'HomeNest'

...

,n_posts,avg_likes,total_shares,avg_followers
brand,,,,
SnackBox,60,2688.8,2949.0,121033.2
Glowly,62,1639.6,1773.0,85566.2
FitFuel,56,1179.4,1230.0,63713.4
UrbanThread,56,841.9,842.0,42027.3
HomeNest,66,459.2,547.0,27272.5


,platform,post_type,likes
8,TikTok,video,2913.0
6,TikTok,carousel,2445.9
5,Instagram,video,1573.5


brand
SnackBox       2.36
UrbanThread    2.19
Glowly         2.03
FitFuel        2.02
HomeNest       1.77
Name: eng_rate, dtype: float64

,post_id,brand,likes,vs_brand
18,P0019,SnackBox,13577,5.05
102,P0103,SnackBox,12052,4.48
179,P0180,HomeNest,2032,4.42


brand        sponsored
FitFuel      False        1150.9
             True         1250.5
Glowly       False        1580.2
             True         1915.1
HomeNest     False         487.6
             True          389.1
SnackBox     False        2936.8
             True         1792.0
UrbanThread  False         850.1
             True          817.4
Name: likes, dtype: float64

In [79]:
# df.groupby("brand").mean() #value column not defined
df.groupby("brand")["likes"].mean()

In [80]:
# df["brand_avg"] = df.groupby("brand")["likes"].mean()
#creates a grouped avg table-- should have transform to add it as a column
#returns 5 rows
# D2: the reason is half right. Look closely: your cell didn’t crash. pandas matched the result’s labels (brand names) against df’s row labels (0–299). Nothing matched, so all 300 values silently became NaN. That’s worse than an error. The fix is your transform line from B8.

In [86]:
df.groupby("brand")["comments"].count() #comments have nans, count counts the non missing stuff only
df.groupby("brand").size()

brand
FitFuel        53
Glowly         61
HomeNest       64
SnackBox       60
UrbanThread    52
Name: comments, dtype: int64

brand
FitFuel        56
Glowly         62
HomeNest       66
SnackBox       60
UrbanThread    56
dtype: int64

In [97]:
# df['platform_avg']= df.groupby('platform')['likes'].transform('mean')
# (df['likes']>df['platform_avg']).sum()

np.int64(114)